# v6 — Vision-OPD-4B 4bit QLoRA (RTX 5060 Ti / CUDA 12.8)

목표는 하나입니다.

> 기존 최고점 `Qwen2.5-VL-3B + QLoRA = 0.92582`와 동일한 데이터/split/객관식 평가 방식에서  
> `Vision-OPD-4B`가 더 나은 fine-grained visual understanding을 보이는지 확인합니다.

이번 버전은 **LLaMA-Factory를 사용하지 않습니다.**
`Transformers + bitsandbytes + PEFT`로 직접 4bit QLoRA를 수행하므로 MiniCPM 실험에서 필요했던
LLaMA-Factory 패치가 필요 없습니다.

### 실행 순서

1. 아래 셀을 위에서부터 실행
2. 처음에는 `TRAIN_LIMIT = 300`으로 smoke test
3. `Validation accuracy`까지 정상 출력되는지 확인
4. 정상이라면 커널 재시작 후 `TRAIN_LIMIT = None`으로 바꾸고 위에서부터 다시 실행
5. 전체 학습일 때만 test submission을 생성

**주의:** 의존성이 이미 MiniCPM 실험으로 맞춰져 있다면 `INSTALL_DEPS=False` 그대로 사용하세요.

In [ ]:
from pathlib import Path
import os, re, math, random, json, sys, subprocess, shutil, gc

import numpy as np
import pandas as pd

ROOT = Path.cwd()
DATA_DIR = ROOT / "dataset"
OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 모델
# --------------------------------------------------
OFFLINE = False

LOCAL_MODEL_DIR = ROOT / "downloads" / "models" / "Vision-OPD-4B"
MODEL_ID = (
    str(LOCAL_MODEL_DIR)
    if OFFLINE
    else "yuanqianhao/Vision-OPD-4B"
)

# --------------------------------------------------
# 실험
# --------------------------------------------------
SEED = 42
VAL_FRAC = 0.10

# 처음에는 300으로 끝까지 돌아가는지만 확인.
# 실제 성능 측정/제출은 None으로 바꾸고 재실행.
TRAIN_LIMIT = 300

EPOCHS = 1
GRAD_ACCUM = 8
LR = 5e-5
WEIGHT_DECAY = 0.01

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

RUN_NAME = "v6_vision_opd_4b"
RUN_DIR = OUTPUT_DIR / RUN_NAME
ADAPTER_DIR = RUN_DIR / "adapter"
RUN_DIR.mkdir(parents=True, exist_ok=True)

if OFFLINE:
    assert LOCAL_MODEL_DIR.exists(), f"로컬 모델이 없습니다: {LOCAL_MODEL_DIR}"

assert DATA_DIR.exists(), f"데이터 폴더가 없습니다: {DATA_DIR}"
assert (DATA_DIR / "train.csv").exists(), "data/train.csv가 없습니다."
assert (DATA_DIR / "test.csv").exists(), "data/test.csv가 없습니다."

print("MODEL_ID   :", MODEL_ID)
print("DATA_DIR   :", DATA_DIR.resolve())
print("RUN_DIR    :", RUN_DIR.resolve())
print("TRAIN_LIMIT:", TRAIN_LIMIT)

## 0. 환경 확인

현재 MiniCPM 실험에서 맞춘 CUDA 환경을 그대로 사용합니다.

권장 기준:

- Python 3.12
- PyTorch `2.11.0+cu128`
- CUDA 12.8
- Transformers `>=5.5` (현재 환경은 5.7.0 권장)
- PEFT `>=0.18`
- bitsandbytes `>=0.48`

`INSTALL_DEPS=True`는 torch를 다시 설치하지 않고 상위 라이브러리만 맞춥니다.

In [ ]:
INSTALL_DEPS = False

if INSTALL_DEPS:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-U",
        "transformers==5.7.0",
        "accelerate==1.13.0",
        "peft>=0.18.1",
        "bitsandbytes>=0.48.2",
        "pillow",
        "pandas",
        "safetensors",
        "sentencepiece",
        "packaging",
    ])
    print("설치 완료. 이 셀을 실행했다면 VSCode에서 Restart Kernel 후 맨 위부터 다시 실행하세요.")
else:
    print("의존성 설치를 건너뜁니다.")

In [ ]:
import torch
import transformers
import accelerate
import peft
import bitsandbytes as bnb
from packaging.version import Version

print("Python      :", sys.version.split()[0])
print("Executable  :", sys.executable)
print("torch       :", torch.__version__)
print("torch CUDA  :", torch.version.cuda)
print("transformers:", transformers.__version__)
print("accelerate  :", accelerate.__version__)
print("peft        :", peft.__version__)
print("bnb         :", bnb.__version__)
print("CUDA usable :", torch.cuda.is_available())

assert torch.cuda.is_available(), "CUDA GPU를 찾지 못했습니다."
assert Version(transformers.__version__) >= Version("5.5.0"), "Vision-OPD-4B는 transformers>=5.5 필요"
assert Version(bnb.__version__) >= Version("0.48.2"), "bitsandbytes>=0.48.2 권장"

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)

print("GPU         :", GPU_NAME)
print("VRAM        :", f"{VRAM_GB:.1f} GB")
print("Capability  :", torch.cuda.get_device_capability(0))

# 8GB 계열이면 보수적으로, 16GB 계열이면 v4와 비슷한 visual token budget 사용
MIN_PIXELS = 256 * 28 * 28
MAX_VISUAL_TOKENS = 384 if VRAM_GB < 12 else 768
MAX_PIXELS = MAX_VISUAL_TOKENS * 28 * 28

print("MIN_PIXELS  :", MIN_PIXELS)
print("MAX_PIXELS  :", MAX_PIXELS, f"(~{MAX_VISUAL_TOKENS} visual-token budget)")

## 1. 데이터 + v4와 동일한 group split

동일/유사 question이 train/valid 양쪽에 겹치지 않도록 기존 v4의 group split 로직을 그대로 사용합니다.
`TRAIN_LIMIT=300`일 때는 smoke test용 소표본이고, `None`일 때가 실제 비교 대상입니다.

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import Any
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

if TRAIN_LIMIT is not None:
    train_df = (
        train_df
        .sample(n=min(TRAIN_LIMIT, len(train_df)), random_state=SEED)
        .reset_index(drop=True)
    )

def normalize_question(x):
    return re.sub(r"\s+", " ", str(x).strip().lower())

def make_group_split(df, val_frac=0.10, seed=42, trials=100):
    groups = {}
    for idx, q in enumerate(df["question"].map(normalize_question)):
        groups.setdefault(q, []).append(idx)

    group_items = list(groups.items())
    target_n = int(round(len(df) * val_frac))
    overall = (
        df["answer"]
        .astype(str).str.lower()
        .value_counts(normalize=True)
        .reindex(list("abcd"), fill_value=0.0)
    )

    best = None
    for t in range(trials):
        rng = random.Random(seed + t)
        items = group_items.copy()
        rng.shuffle(items)

        val_idx = []
        for _, idxs in items:
            if len(val_idx) >= target_n:
                break
            val_idx.extend(idxs)

        val_idx = sorted(set(val_idx))
        val_prop = (
            df.iloc[val_idx]["answer"]
            .astype(str).str.lower()
            .value_counts(normalize=True)
            .reindex(list("abcd"), fill_value=0.0)
        )
        score = float((val_prop - overall).abs().sum()) + abs(len(val_idx) - target_n) / len(df)

        if best is None or score < best[0]:
            best = (score, val_idx)

    val_idx = set(best[1])
    train_idx = [i for i in range(len(df)) if i not in val_idx]
    val_idx = sorted(val_idx)

    return (
        df.iloc[train_idx].reset_index(drop=True),
        df.iloc[val_idx].reset_index(drop=True),
    )

train_subset, valid_subset = make_group_split(train_df, VAL_FRAC, SEED)

print("train total :", len(train_df))
print("train split :", len(train_subset))
print("valid split :", len(valid_subset))
print("test        :", len(test_df))
print("valid dist  :", valid_subset["answer"].astype(str).str.lower().value_counts(normalize=True).sort_index().to_dict())

# 이후 v4/v6 score를 정확히 맞춰볼 수 있도록 validation row 자체를 저장
valid_subset.to_csv(RUN_DIR / "valid_split.csv", index=False)

## 2. 프롬프트

Vision-OPD-4B는 Qwen3.5-4B 기반이며 원 모델은 non-thinking 모드로 학습되었습니다.
따라서 `enable_thinking=False`를 명시해 바로 `a/b/c/d`를 출력하도록 합니다.

In [ ]:
SYSTEM_INSTRUCT = (
    "이미지를 보고 객관식 질문에 답하세요. "
    "필요하면 이미지 속 작은 글자, 숫자, 표지판, 가격, 상호명과 세부 시각 단서를 주의 깊게 확인하세요. "
    "최종 답은 반드시 a, b, c, d 중 하나의 소문자 한 글자만 출력하세요."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"질문: {question}\n"
        f"(a) {a}\n"
        f"(b) {b}\n"
        f"(c) {c}\n"
        f"(d) {d}\n"
        "정답:"
    )

## 3. Vision-OPD-4B 4bit 로드 + Language-only QLoRA

Qwen3.5는 full-attention 레이어와 linear-attention 레이어가 섞여 있습니다.
따라서 기존 Qwen2.5의 `q/k/v/o + MLP`뿐 아니라 Qwen3.5 linear-attention projection도 LoRA 대상으로 포함합니다.

단, module name에 `vision / visual / merger`가 들어가는 모듈은 제외해서 **언어 백본만 학습**합니다.

In [ ]:
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("compute dtype:", COMPUTE_DTYPE)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    local_files_only=OFFLINE,
)
processor.tokenizer.padding_side = "right"

print("processor loaded")

base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=COMPUTE_DTYPE,
    attn_implementation="sdpa",
    local_files_only=OFFLINE,
)

if hasattr(base_model.config, "use_cache"):
    base_model.config.use_cache = False

base_model = prepare_model_for_kbit_training(
    base_model,
    use_gradient_checkpointing=True,
)

try:
    base_model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
except TypeError:
    base_model.gradient_checkpointing_enable()

# Qwen3.5 hybrid text backbone:
# - full attention: q/k/v/o_proj
# - linear attention: in_proj_qkv/in_proj_z/in_proj_a/in_proj_b/out_proj
# - MLP: gate/up/down_proj
TARGET_SUFFIXES = (
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
    "in_proj_qkv", "in_proj_z", "in_proj_a", "in_proj_b", "out_proj",
)

EXCLUDE_TAGS = ("vision", "visual", "merger")

target_modules = []
for name, module in base_model.named_modules():
    lname = name.lower()
    if any(tag in lname for tag in EXCLUDE_TAGS):
        continue
    if name.endswith(TARGET_SUFFIXES):
        target_modules.append(name)

target_modules = sorted(set(target_modules))

if not target_modules:
    raise RuntimeError("LoRA target module을 찾지 못했습니다. Qwen3.5 module naming을 확인하세요.")

print("LoRA target count:", len(target_modules))
print("LoRA target sample:")
for x in target_modules[:30]:
    print("  ", x)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=target_modules,
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# vision LoRA가 실수로 포함되지 않았는지 방어 확인
bad_trainable = [
    name for name, p in model.named_parameters()
    if p.requires_grad and any(tag in name.lower() for tag in EXCLUDE_TAGS)
]
if bad_trainable:
    print("WARNING: vision/projector trainable params 발견:")
    print("\n".join(bad_trainable[:30]))
    for name, p in model.named_parameters():
        if any(tag in name.lower() for tag in EXCLUDE_TAGS):
            p.requires_grad = False
    print("vision/projector params를 강제로 freeze했습니다.")

MODEL_DEVICE = torch.device("cuda:0")
print("model input device:", MODEL_DEVICE)

gc.collect()
torch.cuda.empty_cache()
print("allocated GB:", torch.cuda.memory_allocated() / 1024**3)
print("reserved  GB:", torch.cuda.memory_reserved() / 1024**3)

## 4. Dataset / answer-only loss collator

system/user/image token에는 loss를 주지 않고 **마지막 assistant 정답 토큰만 supervision**합니다.
이 부분은 0.92582가 나온 v4 전략을 유지합니다.

In [ ]:
class VQAMCDataset(Dataset):
    def __init__(self, df, train=True):
        self.df = df.reset_index(drop=True)
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]

        image_path = DATA_DIR / str(row["path"])
        with Image.open(image_path) as im:
            img = im.convert("RGB")

        user_text = build_mc_prompt(
            str(row["question"]),
            str(row["a"]),
            str(row["b"]),
            str(row["c"]),
            str(row["d"]),
        )

        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_INSTRUCT}],
            },
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": user_text},
                ],
            },
        ]

        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({
                "role": "assistant",
                "content": [{"type": "text", "text": gold}],
            })

        return {"messages": messages, "image": img}


def _find_last_subsequence(seq, pattern):
    n = len(pattern)
    for i in range(len(seq) - n, -1, -1):
        if seq[i:i+n] == pattern:
            return i
    return -1


@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __post_init__(self):
        self.assistant_prefix_ids = self.processor.tokenizer.encode(
            "<|im_start|>assistant\n",
            add_special_tokens=False,
        )
        print("assistant prefix ids:", self.assistant_prefix_ids)

    def __call__(self, batch):
        texts, images = [], []

        for sample in batch:
            text = self.processor.apply_chat_template(
                sample["messages"],
                tokenize=False,
                add_generation_prompt=False,
                enable_thinking=False,
            )
            texts.append(text)
            images.append(sample["image"])

        enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        if self.train:
            labels = torch.full_like(enc["input_ids"], -100)

            for i in range(len(batch)):
                valid_len = int(enc["attention_mask"][i].sum().item())
                seq = enc["input_ids"][i, :valid_len].tolist()

                pos = _find_last_subsequence(seq, self.assistant_prefix_ids)
                if pos < 0:
                    decoded = self.processor.tokenizer.decode(seq[-100:])
                    raise RuntimeError(
                        "assistant prefix를 찾지 못했습니다.\n"
                        f"tail={decoded}"
                    )

                answer_start = pos + len(self.assistant_prefix_ids)
                labels[i, answer_start:valid_len] = enc["input_ids"][i, answer_start:valid_len]

            enc["labels"] = labels

        return enc


train_ds = VQAMCDataset(train_subset, train=True)
valid_ds = VQAMCDataset(valid_subset, train=True)

train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True,
    collate_fn=DataCollator(processor, True),
    num_workers=0,
    pin_memory=True,
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=1,
    shuffle=False,
    collate_fn=DataCollator(processor, True),
    num_workers=0,
    pin_memory=True,
)

# 한 배치만 검사
sample_batch = next(iter(train_loader))
print("batch keys:", sample_batch.keys())
print("input shape:", tuple(sample_batch["input_ids"].shape))
print("supervised tokens:", int((sample_batch["labels"] != -100).sum()))
del sample_batch
gc.collect()
torch.cuda.empty_cache()

## 5. 1 epoch QLoRA 학습

- batch size = 1
- gradient accumulation = 8
- LoRA r=8 / alpha=16
- answer-only loss
- gradient checkpointing
- BF16 우선

OOM이면 `MAX_VISUAL_TOKENS`가 자동으로 384로 내려가지만, 그래도 부족하면 위 환경 셀에서
`MAX_VISUAL_TOKENS = 256`으로 고정하고 다시 시작하세요.

In [ ]:
trainable_params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
num_training_steps = EPOCHS * updates_per_epoch

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(num_training_steps * 0.05)),
    num_training_steps=num_training_steps,
)

use_scaler = COMPUTE_DTYPE == torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)

optimizer.zero_grad(set_to_none=True)
global_step = 0

for epoch in range(EPOCHS):
    model.train()
    running_raw_loss = 0.0
    accum_count = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1} [train]",
        unit="batch",
    )

    for step, batch in enumerate(progress_bar, start=1):
        batch = {
            k: (v.to(MODEL_DEVICE, non_blocking=True) if torch.is_tensor(v) else v)
            for k, v in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=COMPUTE_DTYPE,
        ):
            outputs = model(**batch)
            raw_loss = outputs.loss
            loss = raw_loss / GRAD_ACCUM

        scaler.scale(loss).backward()

        running_raw_loss += float(raw_loss.detach().cpu())
        accum_count += 1

        do_update = (
            step % GRAD_ACCUM == 0
            or step == len(train_loader)
        )

        if do_update:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

            global_step += 1

            progress_bar.set_postfix({
                "loss": f"{running_raw_loss / max(accum_count, 1):.4f}",
                "lr": f"{scheduler.get_last_lr()[0]:.2e}",
            })

            running_raw_loss = 0.0
            accum_count = 0

    # answer-only validation loss
    model.eval()
    val_loss = 0.0
    val_steps = 0

    with torch.no_grad():
        for vb in tqdm(
            valid_loader,
            desc=f"Epoch {epoch+1} [valid-loss]",
            unit="batch",
        ):
            vb = {
                k: (v.to(MODEL_DEVICE, non_blocking=True) if torch.is_tensor(v) else v)
                for k, v in vb.items()
            }

            with torch.autocast(
                device_type="cuda",
                dtype=COMPUTE_DTYPE,
            ):
                out = model(**vb)

            val_loss += float(out.loss.detach().cpu())
            val_steps += 1

    print(
        f"[Epoch {epoch+1}] "
        f"answer-only valid loss = {val_loss / max(val_steps, 1):.4f}"
    )

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)

print("adapter saved:", ADAPTER_DIR)

run_config = {
    "model_id": MODEL_ID,
    "train_limit": TRAIN_LIMIT,
    "seed": SEED,
    "val_frac": VAL_FRAC,
    "epochs": EPOCHS,
    "grad_accum": GRAD_ACCUM,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "max_pixels": MAX_PIXELS,
    "max_visual_tokens": MAX_VISUAL_TOKENS,
    "transformers": transformers.__version__,
    "torch": torch.__version__,
    "gpu": GPU_NAME,
}
(RUN_DIR / "run_config.json").write_text(
    json.dumps(run_config, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

## 6. Validation — a/b/c/d 첫 토큰 logits

문자열 `generate()`를 파싱하지 않고 assistant 첫 토큰에서 `a/b/c/d`의 logits을 직접 비교합니다.

추가로 각 validation 샘플의 4개 logits를 CSV에 저장하므로 나중에 v4와 ensemble / 오답 overlap을 볼 수 있습니다.

In [ ]:
CHOICES = ["a", "b", "c", "d"]

choice_token_ids = []
for c in CHOICES:
    ids = processor.tokenizer.encode(c, add_special_tokens=False)
    if len(ids) != 1:
        raise RuntimeError(
            f"'{c}'가 단일 토큰이 아닙니다: {ids}. "
            "sequence scoring이 필요합니다."
        )
    choice_token_ids.append(ids[0])

choice_token_ids = torch.tensor(
    choice_token_ids,
    device=MODEL_DEVICE,
)

print("choice token ids:", dict(zip(CHOICES, choice_token_ids.tolist())))


def build_infer_messages(row, img):
    user_text = build_mc_prompt(
        row["question"],
        row["a"],
        row["b"],
        row["c"],
        row["d"],
    )

    return [
        {
            "role": "system",
            "content": [{"type": "text", "text": SYSTEM_INSTRUCT}],
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": user_text},
            ],
        },
    ]


@torch.inference_mode()
def score_df(df, has_answer=True):
    model.eval()

    if hasattr(model.config, "use_cache"):
        model.config.use_cache = True

    rows = []

    for idx, row in tqdm(
        df.reset_index(drop=True).iterrows(),
        total=len(df),
        desc="Scoring",
        unit="sample",
    ):
        with Image.open(DATA_DIR / str(row["path"])) as im:
            img = im.convert("RGB")

        messages = build_infer_messages(row, img)

        text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )

        inputs = processor(
            text=[text],
            images=[img],
            padding=True,
            return_tensors="pt",
        )

        inputs = {
            k: (v.to(MODEL_DEVICE, non_blocking=True) if torch.is_tensor(v) else v)
            for k, v in inputs.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=COMPUTE_DTYPE,
        ):
            logits = model(**inputs).logits

        last_idx = inputs["attention_mask"].sum(dim=1) - 1
        next_token_logits = logits[0, last_idx.item()]
        choice_logits = (
            next_token_logits
            .index_select(0, choice_token_ids)
            .float()
            .cpu()
            .numpy()
        )

        pred_i = int(choice_logits.argmax())

        rec = {
            "row_idx": idx,
            "id": row["id"] if "id" in row.index else idx,
            "path": row["path"],
            "pred": CHOICES[pred_i],
            "logit_a": float(choice_logits[0]),
            "logit_b": float(choice_logits[1]),
            "logit_c": float(choice_logits[2]),
            "logit_d": float(choice_logits[3]),
        }

        if has_answer:
            rec["answer"] = str(row["answer"]).strip().lower()
            rec["correct"] = rec["pred"] == rec["answer"]

        rows.append(rec)

    return pd.DataFrame(rows)


valid_scores = score_df(valid_subset, has_answer=True)
valid_acc = float(valid_scores["correct"].mean())

print("=" * 60)
print(f"Validation accuracy: {valid_acc:.5f}")
print("baseline v4 LB reference: 0.92582")
print("=" * 60)

VALID_SCORE_PATH = RUN_DIR / "v6_valid_scores.csv"
valid_scores.to_csv(VALID_SCORE_PATH, index=False)
print("saved:", VALID_SCORE_PATH)

display(valid_scores.head())

## 7. Test + submission

`TRAIN_LIMIT=300`은 smoke test이므로 **submission을 만들지 않습니다.**

전체 학습(`TRAIN_LIMIT=None`)일 때만 test를 scoring하고 제출 파일을 생성합니다.

In [ ]:
if TRAIN_LIMIT is not None:
    print(
        "SMOKE TEST 완료입니다. submission은 생성하지 않습니다.\n"
        "Validation까지 정상 동작했다면 TRAIN_LIMIT=None으로 바꾸고 "
        "커널 재시작 후 맨 위부터 다시 실행하세요."
    )
else:
    test_scores = score_df(test_df, has_answer=False)

    TEST_SCORE_PATH = RUN_DIR / "v6_test_scores.csv"
    test_scores.to_csv(TEST_SCORE_PATH, index=False)

    sample_path = DATA_DIR / "sample_submission.csv"

    if sample_path.exists():
        submission = pd.read_csv(sample_path)
        submission["answer"] = test_scores["pred"].values
    else:
        submission = pd.DataFrame({
            "id": test_df["id"].values,
            "answer": test_scores["pred"].values,
        })

    SUBMISSION_PATH = RUN_DIR / "submission_v6_vision_opd_4b.csv"
    submission.to_csv(SUBMISSION_PATH, index=False)

    print("prediction distribution:")
    print(submission["answer"].value_counts(normalize=True).sort_index())
    print("saved:", SUBMISSION_PATH)

    display(submission.head())

## 결과 해석

첫 비교에서는 아래만 봅니다.

- v4 LB 기준: `0.92582`
- v6 validation accuracy
- 이후 실제 Kaggle LB

### 판단

- v6가 v4 근처 또는 상회 → Vision-OPD 유지, 다음으로 해상도/LoRA 범위 실험
- v6 단독이 약간 낮지만 오답 패턴이 다름 → v4 + v6 logits ensemble 후보
- v6가 크게 낮음 → Vision-OPD 추가 튜닝은 중단하고 Qwen3.5-4B 원본 또는 다른 4B 모델로 이동

`output/v6_vision_opd_4b/v6_valid_scores.csv`에는 a/b/c/d logits가 저장됩니다.